# PatrolIQ - Notebook 6: MLflow Experiment Tracking

**Goal:** Use MLflow to keep an organized record of all our clustering and PCA
experiments - their parameters and evaluation scores - instead of just printing
results to the screen and losing track of them.

**Why MLflow:** When comparing multiple algorithms (like we did with K-Means,
DBSCAN, and Hierarchical Clustering), it's easy to lose track of which parameters
gave which results, especially if we go back and re-tune things later. MLflow logs
each "run" so we can always look back and compare.


In [1]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

## Step 1: Set Up MLflow

**Why we use a local SQLite file:** MLflow needs a place to store its tracking
data. Setting up a separate tracking server is more than we need for a student
project - a local SQLite database (`mlflow.db`) is the simplest supported option
and works well for a single-person project like this one.

*(Note: earlier versions of MLflow allowed storing runs directly in a folder
called `mlruns`, but our installed MLflow version requires the SQLite backend
instead - so we use `sqlite:///../mlflow.db`.)*

In [2]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("PatrolIQ_Crime_Clustering")

df = pd.read_csv("../outputs/cleaned_crime_data.csv")

day_map = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3,
           "Friday": 4, "Saturday": 5, "Sunday": 6}
df["day_of_week_num"] = df["day_of_week"].map(day_map)

geo_features = df[["latitude", "longitude"]].values
scaler_geo = StandardScaler()
geo_scaled = scaler_geo.fit_transform(geo_features)

SIL_SAMPLE = 5000

2026/08/02 16:34:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/08/02 16:34:45 INFO mlflow.store.db.utils: Updating database tables


2026/08/02 16:34:47 INFO mlflow.tracking.fluent: Experiment with name 'PatrolIQ_Crime_Clustering' does not exist. Creating a new experiment.


## Step 2: Log the K-Means Geographic Clustering Run

For each experiment, we use `mlflow.start_run()` and then log:
- **Parameters** (`mlflow.log_param`) - the settings we chose, like number of clusters.
- **Metrics** (`mlflow.log_metric`) - the evaluation scores we got, like silhouette score.

In [3]:
with mlflow.start_run(run_name="KMeans_Geographic"):
    k = 7
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(geo_scaled)
    sil = silhouette_score(geo_scaled, labels, sample_size=SIL_SAMPLE, random_state=42)
    db = davies_bouldin_score(geo_scaled, labels)

    mlflow.log_param("algorithm", "KMeans")
    mlflow.log_param("n_clusters", k)
    mlflow.log_param("features", "latitude, longitude")
    mlflow.log_metric("silhouette_score", sil)
    mlflow.log_metric("davies_bouldin_score", db)

    print(f"Logged KMeans run: silhouette={sil:.3f}, db={db:.3f}")

Logged KMeans run: silhouette=0.400, db=0.852


## Step 3: Log the DBSCAN Geographic Clustering Run

In [4]:
with mlflow.start_run(run_name="DBSCAN_Geographic"):
    eps, min_samples = 0.1, 20
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(geo_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)

    mlflow.log_param("algorithm", "DBSCAN")
    mlflow.log_param("eps", eps)
    mlflow.log_param("min_samples", min_samples)
    mlflow.log_param("features", "latitude, longitude")
    mlflow.log_metric("n_clusters_found", n_clusters)
    mlflow.log_metric("n_noise_points", n_noise)

    if n_clusters > 1:
        mask = labels != -1
        sil = silhouette_score(geo_scaled[mask], labels[mask], sample_size=SIL_SAMPLE, random_state=42)
        db = davies_bouldin_score(geo_scaled[mask], labels[mask])
        mlflow.log_metric("silhouette_score", sil)
        mlflow.log_metric("davies_bouldin_score", db)
        print(f"Logged DBSCAN run: clusters={n_clusters}, silhouette={sil:.3f}, db={db:.3f}")

Logged DBSCAN run: clusters=6, silhouette=0.093, db=0.429


## Step 4: Log the Hierarchical Clustering Run

In [5]:
with mlflow.start_run(run_name="Hierarchical_Geographic"):
    sample_idx = np.random.RandomState(42).choice(len(geo_scaled), size=5000, replace=False)
    geo_sample = geo_scaled[sample_idx]

    k = 7
    hierarchical = AgglomerativeClustering(n_clusters=k)
    labels = hierarchical.fit_predict(geo_sample)
    sil = silhouette_score(geo_sample, labels)
    db = davies_bouldin_score(geo_sample, labels)

    mlflow.log_param("algorithm", "Hierarchical")
    mlflow.log_param("n_clusters", k)
    mlflow.log_param("sample_size", 5000)
    mlflow.log_param("features", "latitude, longitude")
    mlflow.log_metric("silhouette_score", sil)
    mlflow.log_metric("davies_bouldin_score", db)

    print(f"Logged Hierarchical run: silhouette={sil:.3f}, db={db:.3f}")

Logged Hierarchical run: silhouette=0.374, db=0.889


## Step 5: Log the PCA Run

In [6]:
with mlflow.start_run(run_name="PCA_DimensionalityReduction"):
    le1, le2 = LabelEncoder(), LabelEncoder()
    df["primary_type_encoded"] = le1.fit_transform(df["primary_type"])
    df["location_description_encoded"] = le2.fit_transform(df["location_description"])

    feature_columns = [
        "latitude", "longitude", "district", "ward", "community_area",
        "hour", "day_of_week_num", "month", "is_weekend", "crime_severity_score",
        "primary_type_encoded", "location_description_encoded", "arrest", "domestic", "year"
    ]
    X = df[feature_columns].copy()
    X["is_weekend"] = X["is_weekend"].astype(int)
    X["arrest"] = X["arrest"].astype(int)
    X["domestic"] = X["domestic"].astype(int)

    scaler_pca = StandardScaler()
    X_scaled = scaler_pca.fit_transform(X)

    n_components = 6
    pca = PCA(n_components=n_components, random_state=42)
    pca.fit_transform(X_scaled)
    total_variance = sum(pca.explained_variance_ratio_)

    mlflow.log_param("algorithm", "PCA")
    mlflow.log_param("n_components", n_components)
    mlflow.log_param("n_input_features", len(feature_columns))
    mlflow.log_metric("total_variance_explained", total_variance)

    print(f"Logged PCA run: {n_components} components, variance={total_variance:.3f}")

Logged PCA run: 6 components, variance=0.714


## Step 6: View All Logged Experiments

We can query MLflow directly in the notebook to see a summary table of all runs,
without needing to open the separate MLflow dashboard.

In [7]:
runs_df = mlflow.search_runs(experiment_names=["PatrolIQ_Crime_Clustering"])
runs_df[["tags.mlflow.runName", "params.algorithm", "metrics.silhouette_score", "metrics.davies_bouldin_score"]]

,tags.mlflow.runName,params.algorithm,metrics.silhouette_score,metrics.davies_bouldin_score
0,PCA_DimensionalityReduction,PCA,NaN,NaN
1,Hierarchical_Geographic,Hierarchical,0.373880,0.888578
2,DBSCAN_Geographic,DBSCAN,0.092951,0.429195
3,KMeans_Geographic,KMeans,0.400245,0.851716


## Step 7: Viewing the Full MLflow Dashboard

To see the full interactive MLflow UI (with all parameters, metrics, and the
ability to compare runs side by side), run this command in a terminal from the
project folder:

```
mlflow ui --backend-store-uri sqlite:///../mlflow.db
```

Then open the link it prints (usually `http://localhost:5000`) in a browser.